In [21]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re
from deep_translator import GoogleTranslator
from selenium.webdriver.common.keys import Keys

In [22]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [23]:
def get_data(slug_name):
    data_list = []
    url = "https://dea.gov.in/whos-who?field_whos_who_category_tid=All&title=&field_email_feedback_email=&page="
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    # options.headless = True
    translator = GoogleTranslator(target='english')
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    driver.get(url)
    while True:
        try:
            time.sleep(2)
            list1 = driver.find_elements(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr')
            print(len(list1))
            for i in range(3, len(list1)+1):
                list2 = driver.find_elements(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td')
                data_dict = {}
                careerInfoDesignation = ""
                emails = ""
                fax = ""
                telephoneNos = ""
                fullAddress = ""
                additionalInfo = ""
                
                for j in range(1, len(list2)+1):
                    if j==1:
                        fullName = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        print(fullName)
                        if fullName == "Vacant" or fullName == "vacant":
                            fullName = ""
                    if j==2:
                        careerInfoDesignation = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        print(careerInfoDesignation)
                    if j==3:
                        emails = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        emails = emails.replace("[at]", "@").replace("[dot]", ".")
                        print(emails)
                    if j==4:
                        telephoneNos = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        if "FAX:" in telephoneNos:
                            fax = telephoneNos.split("FAX:")[1].strip()
                            telephoneNos = telephoneNos.split("FAX:")[0].strip()
                            print(fax)
                        if " FAX" in telephoneNos:
                            fax = telephoneNos.split("\n")[1].strip()
                            telephoneNos = telephoneNos.split("\n")[0].strip()
                        if "(" in telephoneNos:
                            telephoneNos = telephoneNos.split("(")[0]
                            fax = telephoneNos.split("\n", 1)[-1]
                            telephoneNos = telephoneNos.replace(fax, "")
                        telephoneNos = telephoneNos.replace("\n", ", ")
                    if j==5:
                        inter = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        inter = inter.replace("\n", ", ")
                    if j==6:
                        telephoneNos = telephoneNos + ", " + driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text.replace("\n", ", ")
                        print(telephoneNos)
                    if j==7:
                        room = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                    if j==8:
                        fullAddress = driver.find_element(By.XPATH, f'/html/body/section[3]/div/div/div[2]/div[1]/div[2]/div/div/div[2]/div[2]/div[1]/div/table/tr[{i}]/td[{j}]').text
                        print(fullAddress)
                if telephoneNos.strip() == ",":
                    telephoneNos = ""
                summary = fullName + " is the " + careerInfoDesignation
                if inter.strip() != "" and room.strip() !="":
                    additionalInfo = "Intercom No: " + inter + "; Room No: " + room
                if inter.strip() == "" and room.strip() == "":
                    additionalInfo = ""
                if inter.strip() =="" and room.strip() != "":
                    additionalInfo = "Room No: " + room
                if inter.strip() !="" and room.strip() == "":
                    additionalInfo = "Intercom No: " + inter


                print("*"*50)
                if fullName:
                    data_dict['fullName'] = fullName
                if careerInfoDesignation:
                    data_dict['careerInfoDesignation'] = careerInfoDesignation
                if emails:
                    data_dict['emails'] = emails
                if telephoneNos:
                    data_dict['telephoneNos'] = telephoneNos
                if fax:
                    data_dict['fax'] = fax
                if fullAddress:
                    data_dict['fullAddress'] = fullAddress
                if additionalInfo:
                    data_dict['additionalInfo'] = additionalInfo
                if summary:
                    data_dict['summary'] = summary
                if fullName != "" and careerInfoDesignation != "":
                    data_list.append(data_dict)
            driver.find_element(By.LINK_TEXT, f'next ›').click()
        except:
            break
    driver.quit()
    return data_list

In [24]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

18
Smt. Nirmala Sitharaman
Finance Minister
appointment.fm@gov.in, fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 23793791, 23793792
15, Safdarjung Road, New Delhi
**************************************************
Shri S.S. Nakul
Private Secretary to FM
fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 

**************************************************
Shri Vivek Singh
OSD to FM
fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 

**************************************************
Shri B.N. Bhaskar
Addl. PS to FM
fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 

**************************************************
Shri Karma Sonam Zangpo Lhasungpa
Addl. PS to FM
fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 

**************************************************
Shri Sernya Bhutia
1st PA to FM
fmo@nic.in
23092830
23092510, 23092810, 23094399, 23093868, 

**************************************************
Shri Anil Yadav
Under Secr

Shri Anil kumar
PPS to Adviser(Investment)
anil.kumar74@nic.in
23092519, 

**************************************************
Shri P Venkata Swamy
PPS to Adviser (IER)
pv.swamy@nic.in
23092449, 

**************************************************
18
Shri Naresh Kumar
Steno to Sr. Adviser
naresh.kumar83@nic.in
23093752, 9999783587

**************************************************
DIRECTORS/DEPUTY SECRETARIES/OSD’s
**************************************************
Shri Saju K. Surendran
Director (Economic Division)

, 

**************************************************
Dr Monica Thind
Director(G-20)

, 

**************************************************
Ms Shweta Kumar
Director, Economic Division

, 

**************************************************
Sh. Pawan Kumar
Director (DI)
pawankumar.irs@gov.in
23093556, 9820513272
D-II/ 305, Vinay Marg, Chanakyapuri, New Delhi- 110021
**************************************************
Mr. Prabhu Narayan
Director (FS&CS)
prabhu.narayan@gov.i

Shri Parveen Kumar
Under Secretary (FSRL Division)
parveen.k63@nic.in
23095185, 9810205742

**************************************************
Shri Sudipto Sengupta
Under Secretary (Investment Division)
sudipto.sengupta25@nic.in
23094427, 
58-D, Pocket- A-3, DDA Flats, Kalkaji Extension, New Delhi- 110019.
**************************************************
Shri Sunil Kumar Gupta
Under Secretary (ECO. DIVISION)
sunil.gupta75@nic.in
23095260, 7838679371

**************************************************
Shri Pankaj Gupta
Under Secretary (Economic Division-IES Cadre Unit)
pankaj.gupta66@nic.in, pankaj.gupta@nic.in
23095758, 9868450773
63, Kadambari Apptt. Sector 9, Rohini, New Delhi.110085
**************************************************
Shri Ashish Sharma
Under Secretary (BC Division) (Germany & France)
ashish.sharma70@nic.in
23095073, 9818259502

**************************************************
Shri S.R. Raja
Under Secretary (Budget)
raja.sr@nic.in
23094966, 9811020911

***********

23095064, 9711420665, 9910832156
G-190, Nanakpura, New Delhi- 110021
**************************************************
Ms. Deeksha Supyaal Bisht
AD(IES)
deeksha.bisht@gov.in
, 

**************************************************
Ms. Kanika Wadhawan
DD (FU) (IPP)
kanika.wadhawan@gov.in
011-23701075, 9871054122
G-10, Friends Aptts 49, I.P Extension Delhi-110092
**************************************************
Shri Saurabh Bhargava
AD (FM division)
sauravbhargava@hotmail.com
23095220, 8130358055
A 113, Pragati Vihar, Lodhi Road, New Delhi.
**************************************************
Shri M Rahul
DD (Economic Division)
rahul.m@nic.in
23095237, 9868347531

**************************************************
Shri Kumar Shubham
AD (IER Division)
kumarshubham3413@gmail.com
, 7838660526
SB 201, Hudco Place, ND.
**************************************************
Ms. Munesh Sood
AD (FSDC)
sood.munesh@gov.in
23095745, 

**************************************************
Shri Vijay Kumar
D

Vacant
Section Officer (World Bank Infra. & Rural)

, 

**************************************************
Vacant
Section Officer (World Bank Environment)

, 

**************************************************
Vacant
Section Officer (World Bank Water Sector)

, 

**************************************************
Vacant
Section Officer (IMF)

, 

**************************************************
Vacant
Section Officer (WB Policy & Residual Sector)

, 

**************************************************
Shri Kamlesh Kumar
Section Officer(ADB-I)
kamlesh.kmr79@gov.in
23095157, 9971726599

**************************************************
INVESTMENT DIVISION
**************************************************
Shri Arun Kumar Tyagi
Section Officer(FDI Policy)
ak.tyagi@nic.in
23095104, 9968268368

**************************************************
Shri Harsha Bhowmik
Director (Investment Division) (DI&DE)
harsha.bhowmik@gov.in
23093030, 23092420, 9432584109
B-62 Nivedita Kunj, Sector 1, R.


**************************************************
Vacant
Section Officer(Demand)

, 

**************************************************
18
Vacant
Section Officer(Budget Statistics)

, 

**************************************************
Vacant
Section Officer(Planning & Allocation)

, 

**************************************************
Vacant
Section Officer(Account & Technical Advice)

, 

**************************************************
Vacant
Section Officer(Accounts)

, 

**************************************************
Vacant
Section Officer(Public Debt)

, 

**************************************************
Vacant
Section Officer(Budget (Admin))

, 

**************************************************
Vacant
Section Officer(SD)

23095029, 

**************************************************
Vacant
Section Officer(W&M)

23095174, 

**************************************************
Shri Bipin Kumar
Section Officer (States)
bipin.kumar88@nic.in
23095173, 9971651052

*******

23455833, 9711106254

**************************************************
Amlesh Kumar
Caretaker

23458500, 8860902647

**************************************************
MISCELLENEOUS TELEPHONES
**************************************************
Tea Board


, 

**************************************************
Jr. Engg. (Civil)


, 

**************************************************
Coffee Board


23095962, 

**************************************************
C.S.O


23092035, 

**************************************************
IRCTC Canteen


23095601, 

**************************************************
17
CGHS (North Block)


23095629, 

**************************************************
Night Duty Clerk or Care Taker, DEA


23092453, 

**************************************************
AE (Civil)


23093500, 

**************************************************
R.M.L. Hospital


23365525, 

**************************************************
Parliament section (DoE)


, 

********


**************************************************
Vacant
Section Officer (UN)

, 

**************************************************
Paresh Malakar
Section Officer (OMI)
paresh.malakar@nic.in
23095123, 9810768461

**************************************************
INFRASTRUCTURE SUPPORT & DEVELOPMENT DIVISION
**************************************************
Vacant
Section Officer (NIP)

, 

**************************************************
Vacant
Section Officer (PIU)

, 

**************************************************
Vacant
Section Officer (Energy Unit)

, 

**************************************************
